In [4]:
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option("display.width", 120)
print("Library berhasil diimpor.")
 

Library berhasil diimpor.


In [5]:
documents = [
    "Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Gelombang 2 "
    "Tahun 2025 telah dibuka mulai tanggal 12 Januari 2025! Mahasiswa diharapkan segera "
    "mendaftarkan diri melalui SISTEM AKADEMIK ONLINE sebelum batas waktu berakhir.",

    "Tim pengembang Sistem Informasi Akademik (SIA) Fakultas Teknik telah berhasil "
    "mengembangkan fitur baru berupa dashboard nilai mahasiswa secara real-time. "
    "Fitur ini diharapkan dapat mempermudah proses pemantauan akademik oleh dosen dan mahasiswa.",

    "Pengumuman!!! Seminar Nasional Teknologi Komputer & Kecerdasan Buatan akan dilaksanakan "
    "pada 20-21 Februari 2025 di Auditorium Fakultas Teknik. Pendaftaran peserta dibuka mulai "
    "hari ini dengan kuota terbatas hanya untuk 200 peserta.",

    "Laboratorium Komputer Fakultas Teknik UNM melakukan pembaruan perangkat keras dan "
    "perangkat lunak pada 15 unit komputer guna mendukung praktikum mata kuliah Pemrograman "
    "Lanjut dan Kecerdasan Buatan pada semester genap 2025.",

    "Dalam rangka meningkatkan kualitas pendidikan, program studi Teknik Komputer akan "
    "mengadakan pelatihan penggunaan teknologi Artificial Intelligence (AI) bagi dosen dan "
    "mahasiswa pada tanggal 5 Maret 2025 bertempat di ruang seminar lantai 3.",
]

for i, d in enumerate(documents, 1):
    print(f"Dok-{i}: {d}\n")


Dok-1: Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Gelombang 2 Tahun 2025 telah dibuka mulai tanggal 12 Januari 2025! Mahasiswa diharapkan segera mendaftarkan diri melalui SISTEM AKADEMIK ONLINE sebelum batas waktu berakhir.

Dok-2: Tim pengembang Sistem Informasi Akademik (SIA) Fakultas Teknik telah berhasil mengembangkan fitur baru berupa dashboard nilai mahasiswa secara real-time. Fitur ini diharapkan dapat mempermudah proses pemantauan akademik oleh dosen dan mahasiswa.

Dok-3: Pengumuman!!! Seminar Nasional Teknologi Komputer & Kecerdasan Buatan akan dilaksanakan pada 20-21 Februari 2025 di Auditorium Fakultas Teknik. Pendaftaran peserta dibuka mulai hari ini dengan kuota terbatas hanya untuk 200 peserta.

Dok-4: Laboratorium Komputer Fakultas Teknik UNM melakukan pembaruan perangkat keras dan perangkat lunak pada 15 unit komputer guna mendukung praktikum mata kuliah Pemrograman Lanjut dan Kecerdasan Buatan pada semester genap 2025.

Dok-5: Dalam ran

In [6]:
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()


def preprocess_text(text):
    """
    Pipeline preprocessing teks Bahasa Indonesia.

    Returns
    -------
    tokens_before : list[str]  -> token hasil tokenisasi awal (sebelum stopword/stemming)
    tokens_after  : list[str]  -> token hasil akhir (sesudah stopword removal & stemming)
    clean_text    : str        -> teks bersih hasil akhir (token digabung kembali)
    """

    text = text.lower()

    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens_before = text.split()

    text_no_stopwords = stopword_remover.remove(" ".join(tokens_before))

    text_stemmed = stemmer.stem(text_no_stopwords)
    tokens_after = text_stemmed.split()

    return tokens_before, tokens_after, text_stemmed

print("Fungsi preprocess_text() siap digunakan.")


Fungsi preprocess_text() siap digunakan.


In [7]:
results = []
for i, doc in enumerate(documents, start=1):
    tokens_before, tokens_after, clean_text = preprocess_text(doc)
    results.append({
        "id": i,
        "original": doc,
        "tokens_before": tokens_before,
        "n_before": len(tokens_before),
        "clean_text": clean_text,
        "tokens_after": tokens_after,
        "n_after": len(tokens_after),
    })

print(f"Preprocessing selesai untuk {len(results)} dokumen.")


Preprocessing selesai untuk 5 dokumen.


In [18]:
for r in results:
    print("=" * 70)
    print(f"DOKUMEN {r['id']}")
    print("=" * 70)
    print("[SEBELUM PREPROCESSING]")
    print(r["original"])
    print()
    print("[SESUDAH PREPROCESSING]")
    print(r["clean_text"])
    print()
    print(f"Jumlah token sebelum : {r['n_before']}")
    print(f"Jumlah token sesudah : {r['n_after']}")
    print()

DOKUMEN 1
[SEBELUM PREPROCESSING]
Fakultas Teknik UNM mengumumkan bahwa pendaftaran Kuliah Kerja Nyata (KKN) Gelombang 2 Tahun 2025 telah dibuka mulai tanggal 12 Januari 2025! Mahasiswa diharapkan segera mendaftarkan diri melalui SISTEM AKADEMIK ONLINE sebelum batas waktu berakhir.

[SESUDAH PREPROCESSING]
fakultas teknik unm umum daftar kuliah kerja nyata kkn gelombang tahun buka mulai tanggal januari mahasiswa harap segera daftar diri lalu sistem akademik online batas waktu akhir

Jumlah token sebelum : 30
Jumlah token sesudah : 27

DOKUMEN 2
[SEBELUM PREPROCESSING]
Tim pengembang Sistem Informasi Akademik (SIA) Fakultas Teknik telah berhasil mengembangkan fitur baru berupa dashboard nilai mahasiswa secara real-time. Fitur ini diharapkan dapat mempermudah proses pemantauan akademik oleh dosen dan mahasiswa.

[SESUDAH PREPROCESSING]
tim kembang sistem informasi akademik sia fakultas teknik hasil kembang fitur baru upa dashboard nilai mahasiswa real time fitur harap mudah proses pantau

In [19]:
stat_rows = []
for r in results:
    reduction_pct = (r["n_before"] - r["n_after"]) / r["n_before"] * 100
    stat_rows.append({
        "Dokumen": f"Dok-{r['id']}",
        "Token Sebelum": r["n_before"],
        "Token Sesudah": r["n_after"],
        "Pengurangan (%)": round(reduction_pct, 2),
    })

df_stat = pd.DataFrame(stat_rows)
total_before = df_stat["Token Sebelum"].sum()
total_after = df_stat["Token Sesudah"].sum()
total_pct = round((total_before - total_after) / total_before * 100, 2)
df_stat.loc["TOTAL"] = ["TOTAL", total_before, total_after, total_pct]

print(df_stat.to_string(index=False))


Dokumen  Token Sebelum  Token Sesudah  Pengurangan (%)
  Dok-1             30             27            10.00
  Dok-2             32             26            18.75
  Dok-3             27             22            18.52
  Dok-4             28             23            17.86
  Dok-5             29             23            20.69
  TOTAL            146            121            17.12


## Analisis Singkat

Setelah preprocessing, jumlah token berkurang sekitar 20% (persentasenya berbeda-beda
untuk tiap dokumen). Penyebab utamanya adalah penghapusan *stopword* seperti "yang",
"dan", dan "pada", serta imbuhan yang dibuang saat proses *stemming*.

Menurut saya, pengurangan ini penting untuk sistem temu kembali informasi karena
berbagai bentuk kata disatukan menjadi satu kata dasar. Contohnya "mengembangkan",
"dikembangkan", dan "pengembangan" menjadi "kembang". Dengan begitu, kata yang
maknanya sama tidak dianggap sebagai term yang berbeda saat pembobotan maupun saat
mencocokkan dokumen dengan query.

Hasilnya, indeks dokumen menjadi lebih ringkas, gangguan (noise) dari kata-kata
umum berkurang, dan kemiripan antar dokumen atau antara dokumen dengan query bisa
dihitung dengan lebih akurat. Meski begitu, proses ini tetap punya risiko
*over-stemming*, yaitu kata yang dipotong terlalu jauh sehingga maknanya bisa berubah.
Hal ini perlu diperhatikan.